# SENTINEL Qwen3-4B GRPO Training

Colab-friendly runner for the OpenEnv Hackathon submission. It trains the SENTINEL oversight policy with HF TRL GRPO, Qwen3-4B QLoRA, optional Unsloth acceleration, adaptive curriculum, KL guardrails, and proof-pack evaluation.


## 1. Clone and Install

Run this in a GPU runtime. For the final submission, use the onsite Hugging Face compute credits and keep the generated `outputs/` artifacts.

In [ ]:
!git clone https://github.com/sri11223/openEnv.git
%cd openEnv
!pip install -r requirements-train.txt


## 2. Smoke Test the Environment

This should pass before spending GPU time.

In [ ]:
!python validate.py
!python -m pytest tests/test_training_sync.py -q


## 3. Dry-Run the Pipeline

This checks argument wiring without running the full training job.

In [ ]:
%env MODEL_NAME=unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
%env USE_SENTINEL=1
%env USE_UNSLOTH=1
%env TRAIN_STEPS=8
%env WARM_START_STEPS=2
!python scripts/run_sentinel_training_pipeline.py --dry-run --steps 8 --warm-start-steps 2


## 4. Real Training Run

For the judged run, set your Hugging Face token in Colab secrets or the environment. The target run is 300 GRPO steps after a short warm-start.

In [ ]:
# Optional: uncomment if not using Colab secrets
# import os
# os.environ["HF_TOKEN"] = "hf_..."

%env MODEL_NAME=unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
%env USE_SENTINEL=1
%env USE_UNSLOTH=1
%env TRAIN_STEPS=300
%env WARM_START_STEPS=20
%env NUM_GENERATIONS=4
%env KL_ADAPTIVE=1

!python scripts/run_sentinel_training_pipeline.py --steps 300 --warm-start-steps 20


## 5. Submission Artifacts

These are the judge-facing outputs to link or embed in the README/HF Space: reward curve, monitoring snapshot, held-out eval, tripwire eval, proxy-gap summary, and top failure modes.

In [ ]:
!ls -R outputs/monitoring outputs/reward_curves outputs/evals outputs/proof_pack
!python - <<'PY'
import json, pathlib
for path in [
    'outputs/monitoring/latest_summary.json',
    'outputs/proof_pack/proxy_gap_summary.json',
    'outputs/proof_pack/top_failure_modes.json',
]:
    p = pathlib.Path(path)
    print('\n##', path)
    print(json.dumps(json.loads(p.read_text()), indent=2)[:2000] if p.exists() else 'missing')
PY
